# Sprint 1: Tokenization Analysis

This notebook prepares the first descriptive analysis for the thesis project. The goal is not to extract embeddings yet, but to measure how three BERT-like tokenizers fragment the Turkish words in the AnlamVer semantic similarity dataset.

The analysis covers:

- loading and cleaning AnlamVer,
- converting comma-decimal numeric columns such as `Sim` and `Rel` to floats,
- extracting the unique Turkish words from `W1` and `W2`,
- tokenizing all word pairs with BERTurk, multilingual BERT, and XLM-RoBERTa,
- saving a processed pair-level dataset and thesis-ready summary tables.


## Methodological note

AnlamVer contains Turkish word pairs with human semantic similarity (`Sim`) and relatedness (`Rel`) judgments. Since Turkish is morphologically rich and agglutinative, words may be split into multiple subword tokens by transformer tokenizers. This notebook treats tokenization fragmentation as a descriptive property of the dataset and as preparation for later embedding-composition experiments.

The later embedding experiment will compare pooling strategies such as first-token, last-token, mean, and max pooling. This notebook deliberately stops before embedding extraction.

In [2]:
# Load the libraries used in this notebook.
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer

# Use the current working directory as the project root.
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"

# Create output folders if they do not already exist.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_PATH = PROCESSED_DIR / "anlamver_tokenization_analysis.csv"

pd.set_option("display.max_colwidth", 120)

#check check check
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")
print(f"Table output directory: {TABLE_DIR}")   

/Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect
Raw data directory: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/data/raw
Processed data directory: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/data/processed
Table output directory: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/outputs/tables


## Load AnlamVer

The notebook uses the project dataset at `data/raw/anlamver-final.csv`. The original AnlamVer CSV may use semicolon separators and Turkish/Windows encoding, so the helper tries those settings after a general automatic read attempt.

In [3]:
# Read the main AnlamVer dataset from the known project location.
INPUT_PATH = RAW_DIR / "anlamver-final.csv"

def read_anlamver(path: Path) -> pd.DataFrame:
    """Read AnlamVer with fallback settings for the original CSV format."""
    try:
        return pd.read_csv(path, sep=None, engine="python", encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, sep=";", encoding="windows-1254")
    except pd.errors.ParserError:
        return pd.read_csv(path, sep=";", encoding="windows-1254")

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Could not find the AnlamVer input file: {INPUT_PATH}")

df = read_anlamver(INPUT_PATH)
print(f"Loaded {len(df):,} rows from {INPUT_PATH}")
df.head()


Loaded 500 rows from /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/data/raw/anlamver-final.csv


,QID,W1,W2,Sim,Rel,S,D,R,U,SR,...,EstMer,W1-RWG,W2-RWG,RWMin,W1-DG,W2-DG,DGMax,W1-IG,W2-IG,IGMax
0,1,mızrap,barınak,"0,166666667","0,083333333",False,True,False,True,False,...,0,2,3,2,0,1,1,0,0,0
1,2,kırmızı,gül,"1,166666667","7,166666667",False,True,True,False,False,...,0,5,5,5,0,0,0,0,0,0
2,3,suçlu,şüphe,"2,416666667",7,False,True,True,False,False,...,0,4,4,4,1,0,1,0,0,0
3,4,laikçiler,sekülerizmciler,9,"9,416666667",True,False,True,False,True,...,0,2,0,0,2,0,2,0,0,0
4,5,bitki,zeytin,"3,666666667","6,75",False,True,True,False,False,...,0,4,4,4,0,0,0,0,0,0


## Clean numeric columns

The AnlamVer ratings are stored with comma decimals, for example `0,166666667`. These need to be converted to numeric floats before later correlation analyses. Here the cleaning is limited to columns that are expected to be numeric and present in the dataset.

In [4]:
def comma_decimal_to_float(series: pd.Series) -> pd.Series:
    """Convert strings such as '1,25' to floats while preserving missing values."""
    return pd.to_numeric(
        series.astype("string").str.replace(",", ".", regex=False),
        errors="coerce",
    )

# Convert only the numeric columns that exist in the dataset.
numeric_columns = [
    "Sim", "Rel", "AVG-C", "W1F", "W2F", "RWMin", "DGMax", "IGMax",
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = comma_decimal_to_float(df[column])

df[[column for column in ["W1", "W2", "Sim", "Rel"] if column in df.columns]].head()


,W1,W2,Sim,Rel
0,mızrap,barınak,0.166667,0.083333
1,kırmızı,gül,1.166667,7.166667
2,suçlu,şüphe,2.416667,7.0
3,laikçiler,sekülerizmciler,9.0,9.416667
4,bitki,zeytin,3.666667,6.75


## Extract unique words

The tokenizers are applied to word types first, not directly to every pair occurrence. This avoids repeated work and makes it easy to summarize fragmentation at the unique-word level.

In [5]:
# Make sure the two word columns needed for tokenization are available.
required_columns = {"W1", "W2"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

# Tokenize each unique word once, then reuse those results for word pairs.
unique_words = pd.Index(pd.concat([df["W1"], df["W2"]]).dropna().astype(str).str.strip().unique()).sort_values()

print(f"Word pairs: {len(df):,}")
print(f"Word occurrences: {len(df) * 2:,}")
print(f"Unique words: {len(unique_words):,}")
unique_words[:10].tolist()


Word pairs: 500
Word occurrences: 1,000
Unique words: 317


['adalet',
 'affedişi',
 'ahşap',
 'aile',
 'akıllı',
 'alacaklanmak',
 'alet',
 'alkol',
 'almışlardır',
 'alışkın']

## Tokenizers

Three tokenizers are compared:

- **BERTurk**: `dbmdz/bert-base-turkish-cased`, a Turkish-specific BERT model.
- **mBERT**: `bert-base-multilingual-cased`, a multilingual WordPiece model.
- **XLM-RoBERTa**: `xlm-roberta-base`, a multilingual SentencePiece model.

For each tokenizer, a word is considered *split* when it receives more than one subtoken. A pair is considered split when either `W1` or `W2` is split.

In [6]:
# Define the tokenizers compared in this analysis.
MODELS = {
    "berturk": "dbmdz/bert-base-turkish-cased",
    "mbert": "bert-base-multilingual-cased",
    "xlmr": "xlm-roberta-base",
}

def tokenize_words(model_name: str, words: pd.Index) -> pd.DataFrame:
    """Return one row per unique word with tokenizer tokens and token counts."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    rows = []
    for word in words:
        tokens = tokenizer.tokenize(word)
        rows.append({
            "word": word,
            "tokens": tokens,
            "subtoken_count": len(tokens),
            "is_split": len(tokens) > 1,
        })
    return pd.DataFrame(rows)

# Build one tokenization table per model.
word_tokenizations = {
    short_name: tokenize_words(model_name, unique_words)
    for short_name, model_name in MODELS.items()
}

word_tokenizations["mbert"].head()


,word,tokens,subtoken_count,is_split
0,adalet,"[ada, ##let]",2,True
1,affedişi,"[af, ##fe, ##di, ##şi]",4,True
2,ahşap,"[ah, ##şa, ##p]",3,True
3,aile,[aile],1,False
4,akıllı,"[ak, ##ıl, ##lı]",3,True


In [7]:
word_tokenizations["xlmr"].head()


,word,tokens,subtoken_count,is_split
0,adalet,"[▁ada, let]",2,True
1,affedişi,"[▁aff, edi, şi]",3,True
2,ahşap,"[▁ah, şa, p]",3,True
3,aile,[▁aile],1,False
4,akıllı,[▁akıllı],1,False


## Add tokenizer features to word pairs

The processed pair-level dataset keeps the original AnlamVer columns and adds model-specific tokenization features. Token lists are stored as space-separated strings in the CSV so they remain easy to inspect in spreadsheet software.

In [8]:
# Start from the cleaned AnlamVer data and add tokenizer features.
analysis_df = df.copy()

for short_name, token_df in word_tokenizations.items():
    token_lookup = token_df.set_index("word")

    for side, source_column in [("word1", "W1"), ("word2", "W2")]:
        words = analysis_df[source_column].astype(str).str.strip()
        analysis_df[f"{short_name}_{side}_tokens"] = words.map(token_lookup["tokens"]).apply(" ".join)
        analysis_df[f"{short_name}_{side}_subtoken_count"] = words.map(token_lookup["subtoken_count"]).astype("Int64")

    # Summarize token counts at the word-pair level.
    w1_count = analysis_df[f"{short_name}_word1_subtoken_count"]
    w2_count = analysis_df[f"{short_name}_word2_subtoken_count"]
    analysis_df[f"{short_name}_pair_total_subtoken_count"] = w1_count + w2_count
    analysis_df[f"{short_name}_pair_max_subtoken_count"] = pd.concat([w1_count, w2_count], axis=1).max(axis=1)
    analysis_df[f"{short_name}_pair_clean"] = (w1_count == 1) & (w2_count == 1)
    analysis_df[f"{short_name}_pair_split"] = ~analysis_df[f"{short_name}_pair_clean"]

analysis_columns = [
    "W1", "W2", "Sim", "Rel",
    "berturk_word1_tokens", "berturk_word2_tokens",
    "berturk_word1_subtoken_count", "berturk_word2_subtoken_count",
    "berturk_pair_total_subtoken_count", "berturk_pair_max_subtoken_count",
    "berturk_pair_clean", "berturk_pair_split",
]
analysis_df[[column for column in analysis_columns if column in analysis_df.columns]].head()


,W1,W2,Sim,Rel,berturk_word1_tokens,berturk_word2_tokens,berturk_word1_subtoken_count,berturk_word2_subtoken_count,berturk_pair_total_subtoken_count,berturk_pair_max_subtoken_count,berturk_pair_clean,berturk_pair_split
0,mızrap,barınak,0.166667,0.083333,mız ##rap,barın ##ak,2,2,4,2,False,True
1,kırmızı,gül,1.166667,7.166667,kırmızı,gül,1,1,2,1,True,False
2,suçlu,şüphe,2.416667,7.0,suçlu,şüphe,1,1,2,1,True,False
3,laikçiler,sekülerizmciler,9.0,9.416667,laik ##çiler,sek ##üler ##izm ##ciler,2,4,6,4,False,True
4,bitki,zeytin,3.666667,6.75,bitki,zeytin,1,1,2,1,True,False


## Summary tables

The tables below are designed for the Methods and Data Exploration sections. They describe how often words and pairs are split, how severe the fragmentation is, and which words are most fragmented for each tokenizer.

In [9]:
def subtoken_distribution(tokenizations: dict[str, pd.DataFrame], pair_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize subtoken counts for unique words and word occurrences."""
    rows = []
    for short_name, token_df in tokenizations.items():
        unique_counts = token_df["subtoken_count"].value_counts().sort_index()
        occurrence_counts = pd.concat([
            pair_df[f"{short_name}_word1_subtoken_count"],
            pair_df[f"{short_name}_word2_subtoken_count"],
        ]).value_counts().sort_index()

        all_counts = sorted(set(unique_counts.index).union(set(occurrence_counts.index)))
        for subtoken_count in all_counts:
            unique_word_count = int(unique_counts.get(subtoken_count, 0))
            word_occurrence_count = int(occurrence_counts.get(subtoken_count, 0))
            rows.append({
                "model": short_name,
                "subtoken_count": int(subtoken_count),
                "unique_words": unique_word_count,
                "unique_word_percentage": round(unique_word_count / len(token_df) * 100, 2),
                "word_occurrences": word_occurrence_count,
                "word_occurrence_percentage": round(word_occurrence_count / (len(pair_df) * 2) * 100, 2),
            })
    return pd.DataFrame(rows)

def split_pair_percentages(pair_df: pd.DataFrame) -> pd.DataFrame:
    """Calculate how many word pairs contain at least one split word."""
    rows = []
    for short_name in MODELS:
        split_pairs = pair_df[f"{short_name}_pair_split"].sum()
        clean_pairs = pair_df[f"{short_name}_pair_clean"].sum()
        rows.append({
            "model": short_name,
            "word_pairs": len(pair_df),
            "clean_pairs": int(clean_pairs),
            "split_pairs": int(split_pairs),
            "split_pair_percentage": round(split_pairs / len(pair_df) * 100, 2),
        })
    return pd.DataFrame(rows)

def three_plus_token_words(tokenizations: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Count words that are split into three or more subtokens."""
    rows = []
    for short_name, token_df in tokenizations.items():
        n_words = (token_df["subtoken_count"] >= 3).sum()
        rows.append({
            "model": short_name,
            "unique_words": len(token_df),
            "three_plus_token_words": int(n_words),
            "percentage": round(n_words / len(token_df) * 100, 2),
        })
    return pd.DataFrame(rows)

def most_fragmented_words(tokenizations: dict[str, pd.DataFrame], top_n: int = 25) -> pd.DataFrame:
    """List the most heavily split words for each tokenizer."""
    frames = []
    for short_name, token_df in tokenizations.items():
        top_words = token_df.sort_values(["subtoken_count", "word"], ascending=[False, True]).head(top_n).copy()
        top_words.insert(0, "model", short_name)
        top_words["tokens"] = top_words["tokens"].apply(" ".join)
        frames.append(top_words)
    return pd.concat(frames, ignore_index=True)

# Create all summary tables used below and saved at the end.
summary_distribution = subtoken_distribution(word_tokenizations, analysis_df)
summary_split_pairs = split_pair_percentages(analysis_df)
summary_three_plus = three_plus_token_words(word_tokenizations)
summary_most_fragmented = most_fragmented_words(word_tokenizations)


In [10]:
summary_distribution


,model,subtoken_count,unique_words,unique_word_percentage,word_occurrences,word_occurrence_percentage
0,berturk,1,185,58.36,620,62.0
1,berturk,2,83,26.18,244,24.4
2,berturk,3,37,11.67,106,10.6
3,berturk,4,12,3.79,30,3.0
4,mbert,1,31,9.78,110,11.0
5,mbert,2,105,33.12,342,34.2
6,mbert,3,104,32.81,333,33.3
7,mbert,4,51,16.09,150,15.0
8,mbert,5,16,5.05,40,4.0
9,mbert,6,8,2.52,20,2.0


In [11]:
summary_split_pairs


,model,word_pairs,clean_pairs,split_pairs,split_pair_percentage
0,berturk,500,225,275,55.0
1,mbert,500,12,488,97.6
2,xlmr,500,93,407,81.4


In [12]:
summary_three_plus


,model,unique_words,three_plus_token_words,percentage
0,berturk,317,49,15.46
1,mbert,317,181,57.10
2,xlmr,317,94,29.65


In [13]:
summary_most_fragmented.head(30)


,model,word,tokens,subtoken_count,is_split
0,berturk,anarşist,ana ##r ##şi ##st,4,True
1,berturk,asosyalleştirdikleri,as ##osyal ##leştir ##dikleri,4,True
2,berturk,biçimlendirilmişlerdir,biçim ##lendir ##ilmiş ##lerdir,4,True
3,berturk,gençmişçesine,genç ##miş ##çesi ##ne,4,True
4,berturk,kavramsallaştırsın,kavram ##sal ##laştır ##sın,4,True
5,berturk,kemalizmcilerden,kemal ##izm ##ciler ##den,4,True
6,berturk,manavınkiler,mana ##v ##ın ##kiler,4,True
7,berturk,pimpirikli,pi ##m ##pir ##ikli,4,True
8,berturk,primatçaları,prim ##at ##ça ##ları,4,True
9,berturk,saldırmaktaydılar,saldır ##makta ##ydı ##lar,4,True


## Save processed data and tables

The processed dataset is saved at pair level. Summary tables and the reusable unique-word list are saved separately under `outputs/tables/` so they can be reused in the thesis text, slides, or later analysis notebooks.

In [14]:
# Save the pair-level dataset with tokenizer features.
analysis_df.to_csv(PROCESSED_PATH, index=False)

# Save separate summary tables and reusable word list for reporting and later notebooks.
summary_distribution.to_csv(TABLE_DIR / "0101-tokenization_subtoken_count_distribution.csv", index=False)
summary_split_pairs.to_csv(TABLE_DIR / "0102-tokenization_split_pair_percentage.csv", index=False)
summary_three_plus.to_csv(TABLE_DIR / "0103-tokenization_three_plus_token_words.csv", index=False)
summary_most_fragmented.to_csv(TABLE_DIR / "0104-tokenization_most_fragmented_words.csv", index=False)
pd.DataFrame({"word": unique_words}).to_csv(TABLE_DIR / "0105-tokenization_unique_words.csv", index=False)

print(f"Saved processed dataset to: {PROCESSED_PATH}")
print(f"Saved summary tables to: {TABLE_DIR}")


Saved processed dataset to: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/data/processed/anlamver_tokenization_analysis.csv
Saved summary tables to: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/outputs/tables


## Interpretation checklist

Use these points when writing up the Sprint 1 results:

- A high split-pair percentage means many AnlamVer pairs require a subtoken-to-word composition decision before embeddings can be compared.
- The `3+` token subset identifies the words where first-token or last-token pooling is most likely to discard information.
- Differences between BERTurk, mBERT, and XLM-R show whether Turkish-specific tokenization reduces fragmentation compared with multilingual tokenization.
- These statistics are descriptive. Fragmentation should not be interpreted as causing lower human similarity ratings without later modeling and controls.